# Description

This notebook contains the code used to generate Figure 1, which exemplifies how covariance-based FC behaves across echoes in the two extreme signal regimes

In [1]:
from utils.basics import PRCS_DATA_DIR
import os.path as osp
import numpy as np
import pandas as pd
from nilearn.connectome import sym_matrix_to_vec
import hvplot.pandas
import holoviews as hv
import panel as pn

***

# 1. Non-BOLD Behavior

The code below generates Figure 1.a where we show both a simulation and real case of how data that is non-BOLD dominated behaves.

First, we load real non-constant TR data (i.e., one random scan from the discovery dataset)

In [2]:
sbj        = 'MGSBJ03'       # Subject ID 
ses        = 'cardiac_gated' # Session ID
scenario   = 'ALL_Basic'     # Basic denoising
ATLAS_NAME = 'Power264-discovery' # Atlas ID used for the discovery dataset (only ROIs in the FOV for this dataset)
tes        = ['e01','e02','e03']
te_labels  = {'e01':'TE1','e02':'TE2','e03':'TE3'}


## 1.1. Load timeseries for all available TEs:

In [3]:
roi_ts = {}
for te in tes:
    roi_ts_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'errts.{sbj}.r01.{te}.volreg.spc.tproject_{scenario}.{ATLAS_NAME}_000.netts')
    roi_ts[te] = np.loadtxt(roi_ts_path)

## 1.2. Compute the covariance matrix for all pairs of TEs:

In [4]:
FC_c    = {}
for te_x in tes:
    for te_y in tes:
        aux_ts_x = roi_ts[te_x]
        aux_ts_y = roi_ts[te_y]
        aux_fc_c = np.cov(aux_ts_x.T, aux_ts_y.T)[:aux_ts_x.shape[1], aux_ts_y.shape[1]:]
        FC_c[(te_x,te_y)] = sym_matrix_to_vec(aux_fc_c,discard_diagonal=True)

## 1.3 Generate Simulated Scatter Plot

To generate a "cartoonish" scatter plot that exemplifies the non-BOLD behavior--namely points sitting over the identity line--we do the following:

1) ```x_data```: We load real FC_c for a pair of echoes (```(e01,e02)```)
2) ```y_data```: Instead of loading FC_c for a second pair of echoes, we simulate the FC_c for that second pair of echoes by adding a bit of noise to the one loaded in 1.

In [5]:
x_data = FC_c[('e01','e02')]
y_data = x_data + 0.1 * (np.random.normal(0,.25,25425))

Next, we load the two sets of FC_c into a single pandas dataframe for plotting purposes

In [6]:
non_bold_sim_df = pd.DataFrame([x_data,y_data], 
                  index=['C(TEi,TEj)','C(TEk,TEl)']).T

In [7]:
non_bold_sim_scatter = non_bold_sim_df.hvplot.scatter(x='C(TEi,TEj)',y='C(TEk,TEl)', 
                                   aspect='square',
                                   color='black', 
                                   datashade=True, 
                                   xlabel='C(TEi,TEj)', 
                                   ylabel='C(TEk,TEl)', xlim=(-.1,1.5), ylim=(-.1,1.5)).opts(fontscale=1.2, 
                                                                                             title='(a) Simulation', 
                                                                                             active_tools=['reset'])

## 1.4. Generate Real Data Scatter Plot

In this case we load real data for both the x and y axis of the plot

In [8]:
x_data       = FC_c[('e01','e02')]
x_data_label = f"C({te_labels['e01']}"
y_data       = FC_c[('e01','e03')]
y_data_label = f"C({te_labels['e01']},{te_labels['e03']})"

In [9]:
non_bold_real_df = pd.DataFrame([x_data,y_data], 
                                index=[x_data_label, y_data_label]).T



In [10]:
non_bold_real_scatter = non_bold_real_df.hvplot.scatter(x=x_data_label,y=y_data_label, 
                                   aspect='square',
                                   color='black', 
                                   datashade=True, 
                                   xlabel=x_data_label, 
                                   ylabel=y_data_label, xlim=(-.1,1.5), ylim=(-.1,1.5)).opts(fontscale=1.2, 
                                                                                             title='(b) Real Data', 
                                                                                             active_tools=['reset'])


# 1.5 Generate Panels a and b 

First, we create the reference axes for (0,0)

In [11]:
zero_marker = hv.VLine(0).opts(line_width=0.5, line_dash='dashed', line_color='gray') * hv.HLine(0).opts(line_width=0.5, line_dash='dashed', line_color='gray')

Second, we create a red dashed line that has slope = 0 and intersect = 1. This is the expected behavior for the non-BOLD dominant regime

In [12]:
non_BOLD_line = hv.Slope(1,0).opts(line_color='r',line_dash='dashed',line_width=2)

Third, we compose the individual panels by adding the non_BOLD_line and the zero marker to each of the scatter plot

In [13]:
non_bold_sim_panel  = non_bold_sim_scatter * non_BOLD_line * zero_marker
non_bold_real_panel = non_bold_real_scatter * non_BOLD_line * zero_marker

Forth, we combine then into single plot using ```panel.GridBox```

In [14]:
non_bold_banner = pn.pane.HTML(
    """
    <div style="
        width:100%;
        background:#000;
        color:#fff;
        font-weight:700;
        text-align:center;
        padding:10px 12px;
        box-sizing:border-box;
        font-size:18px;
    ">
      Non-BOLD Dominated Data
    </div>
    """,
    sizing_mode="scale_width",
)

***

# 2. BOLD-Dominated Behavior

In [15]:
fig1 = pn.Column(non_bold_banner,pn.Row(non_bold_sim_panel,non_bold_real_panel), width=850)

In [16]:
fig1.save('figures/fig1.html')

In [ ]:
from IPython.display import HTML
HTML(filename="figures/fig1.html")